# Лабораторная №6. Представления узлов графа и классификация вершин

__Автор задач: Пасканов В.Д.__

Цель: сравнить качество решения задачи классификации вершин графа для трёх подходов:

1. Бейзлайн без графовых признаков (например, CatBoost по табличным признакам).
2. Node2Vec с использованием готовой библиотеки.
3. Кастомный метод обхода графа + выбранная модель из NLP.


Материалы:

* Документация:
  * Node2Vec (реализация на Python): https://github.com/eliorc/node2vec
  * Gensim Word2Vec: https://radimrehurek.com/gensim/models/word2vec.html
  * CatBoost: https://catboost.ai/en/docs/
  * PyTorch: https://pytorch.org/docs/stable/index.html
  * HuggingFace Transformers (для идей по архитектурам): https://huggingface.co/docs/transformers/index

* Обзоры методов обхода графа и random walk:
  * Node2Vec: Scalable Feature Learning for Networks (Grover, Leskovec, 2016): https://snap.stanford.edu/node2vec/
  * DeepWalk: Online Learning of Social Representations (Perozzi et al., 2014): https://arxiv.org/abs/1403.6652
  * Role2Vec: https://arxiv.org/abs/1802.02896
  * Biased random walks для node2vec (обзор): https://memgraph.com/blog/how-node2vec-works


In [14]:
import gc
import numpy as np
import pandas as pd
import networkx as nx
import torch
from typing import List, Tuple, Dict, Any
from tqdm import tqdm
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier

from gensim.models import Word2Vec
from node2vec import Node2Vec

from ogb.nodeproppred import NodePropPredDataset
from scipy.sparse import csr_matrix

warnings.filterwarnings('ignore')

## Задачи для самостоятельного решения


### Задание 1. Выбор датасета для классификации узлов

Выберите **графовый датасет для задачи классификации узлов**, в котором:

* число узлов не менее 1000;
* у узлов есть **метки классов** (labels);
* у узлов желательно есть **атрибуты/признаки** (feature matrix), чтобы можно было построить бейзлайн без использования структуры графа.

Примеры источников графовых датасетов:

* PyTorch Geometric Datasets (Cora, Citeseer, Pubmed, Amazon, Reddit и др.):
  * https://pytorch-geometric.readthedocs.io/en/latest/modules/datasets.html
* SNAP (Stanford Network Analysis Project):
  * https://snap.stanford.edu/data/index.html
* Open Graph Benchmark (OGB):
  * https://ogb.stanford.edu/docs/nodeprop/

Рекомендуется взять один из классических датасетов (Cora / Citeseer / Pubmed / OGBN-Arxiv и т.п.), либо другой датасет, который удовлетворяет требованиям по размеру и наличию меток.

Опишите в этом ноутбуке, какой датасет вы выбрали:

* краткое текстовое описание (что за граф, что означают узлы и рёбра);
* число узлов, число рёбер, число классов;
* какие признаки есть у узлов.


**Датасет: OGBN-MAG (Microsoft Academic Graph)**

* **Описание:** Это гетерогенная сеть академических публикаций. 
* **Узлы (4 типа):** `paper` (статья), `author` (автор), `institution` (институт), `field_of_study` (область науки).
* **Рёбра:** автор-пишет-статью, статья-цитирует-статью и т.д.
* **Задача:** Предсказать место публикации (venue) для каждой статьи (349 уникальных классов).
* **Признаки (Features):** Узлы `paper` имеют 128-мерные векторные признаки, полученные с помощью предобученной модели Word2Vec на текстах аннотаций и названий статей.

In [15]:
original_load = torch.load

def patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)

try:
    torch.load = patched_load
    print("Загрузка полного датасета ogbn-mag (кэш)...")
    dataset = NodePropPredDataset(name='ogbn-mag', root='data/')
finally:
    torch.load = original_load

graph = dataset[0][0]
labels = dataset[0][1]

orig_features = graph['node_feat_dict']['paper']
orig_labels = labels['paper'].flatten() 
orig_edge_index = graph['edge_index_dict'][('paper', 'cites', 'paper')]

SUBSET_SIZE = 30000

print(f"\nИзвлечение подграфа на {SUBSET_SIZE} узлов...")

subset_nodes = np.arange(SUBSET_SIZE)

paper_features = orig_features[subset_nodes]
paper_labels = orig_labels[subset_nodes]

edge_mask = (orig_edge_index[0] < SUBSET_SIZE) & (orig_edge_index[1] < SUBSET_SIZE)

edge_index_paper = orig_edge_index[:, edge_mask]

num_papers = paper_features.shape[0]
num_edges = edge_index_paper.shape[1]
num_classes = len(np.unique(paper_labels[~np.isnan(paper_labels)]))

print(f"\n[РЕЗУЛЬТАТЫ СЭМПЛИНГА]")
print(f"Количество узлов (paper): {num_papers:,}")
print(f"Количество рёбер (cites): {num_edges:,}")
print(f"Количество классов: {num_classes}")
print(f"Размерность признаков: {paper_features.shape[1]}")

valid_mask = ~np.isnan(paper_labels)
valid_indices = np.where(valid_mask)[0]

print(f"Количество узлов с известными метками: {len(valid_indices):,}")

Загрузка полного датасета ogbn-mag (кэш)...

Извлечение подграфа на 30000 узлов...

[РЕЗУЛЬТАТЫ СЭМПЛИНГА]
Количество узлов (paper): 30,000
Количество рёбер (cites): 9,904
Количество классов: 344
Размерность признаков: 128
Количество узлов с известными метками: 30,000


### Задание 2. Train/val/test сплит по узлам

Сделайте разбиение узлов на три непересекающихся подмножества:

* `train` — для обучения моделей;
* `val` — для подбора гиперпараметров;
* `test` — для финальной оценки качества.

Рекомендуемые пропорции: 60% / 20% / 20% от числа узлов.

Желательно, чтобы распределение классов в каждом сплите было примерно одинаковым (стратифицированное разбиение по меткам).


In [16]:
y_valid = paper_labels[valid_indices]
class_counts = pd.Series(y_valid).value_counts()
valid_classes = class_counts[class_counts >= 10].index

mask_frequent_classes = np.isin(y_valid, valid_classes)
filtered_valid_indices = valid_indices[mask_frequent_classes]
filtered_y = paper_labels[filtered_valid_indices]

print(f"Было узлов для сплита: {len(valid_indices)}")
print(f"Осталось узлов после удаления 'одиночек': {len(filtered_valid_indices)}")
print(f"Удалено узлов редких классов: {len(valid_indices) - len(filtered_valid_indices)}")

X_train_idx, X_temp_idx, y_train, y_temp = train_test_split(
    filtered_valid_indices, 
    filtered_y, 
    test_size=0.4, 
    random_state=42, 
    stratify=filtered_y
)

X_val_idx, X_test_idx, y_val, y_test = train_test_split(
    X_temp_idx, 
    y_temp, 
    test_size=0.5, 
    random_state=42, 
    stratify=y_temp
)

print(f"\nРазмеры выборок:")
print(f"Train size: {len(X_train_idx):,} ({len(X_train_idx)/len(filtered_valid_indices):.0%})")
print(f"Val size:   {len(X_val_idx):,} ({len(X_val_idx)/len(filtered_valid_indices):.0%})")
print(f"Test size:  {len(X_test_idx):,} ({len(X_test_idx)/len(filtered_valid_indices):.0%})")

Было узлов для сплита: 30000
Осталось узлов после удаления 'одиночек': 29593
Удалено узлов редких классов: 407

Размеры выборок:
Train size: 17,755 (60%)
Val size:   5,919 (20%)
Test size:  5,919 (20%)


### Задание 3. Бейзлайн модель без графа (CatBoost)

Постройте бейзлайн, который **не использует структуру графа**, а работает только с табличными признаками узлов.
В качестве такой модели можно использовать, например, `CatBoostClassifier`.

Шаги:

1. Подготовьте матрицу признаков `X` для узлов и вектор меток `y`.
2. Разделите `X` и `y` на `X_train`, `X_val`, `X_test` согласно сплиту по узлам.
3. Обучите модель CatBoost на `train`, подберите гиперпараметры по `val` (или используйте разумные значения по умолчанию).
4. Оцените качество на `test` (accuracy, F1, macro-F1 и др.).

Сохраните метрики бейзлайна — они понадобятся для сравнения с графовыми методами.


In [17]:
cb_baseline = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    loss_function='MultiClass',
    task_type='GPU',   
    random_seed=42,
    verbose=50
)

X_train_feat = paper_features[X_train_idx]
X_val_feat = paper_features[X_val_idx]
X_test_feat = paper_features[X_test_idx]

cb_baseline.fit(
    X_train_feat, y_train,
    eval_set=(X_val_feat, y_val),
    early_stopping_rounds=20
)

y_pred_test_cb = cb_baseline.predict(X_test_feat)

acc_baseline = accuracy_score(y_test, y_pred_test_cb)
f1_macro_baseline = f1_score(y_test, y_pred_test_cb, average='macro')

print(f"\n[Baseline] Accuracy на test: {acc_baseline:.4f}")
print(f"[Baseline] Macro-F1 на test: {f1_macro_baseline:.4f}")

0:	learn: 5.2525860	test: 5.2744459	best: 5.2744459 (0)	total: 255ms	remaining: 4m 14s
50:	learn: 2.9817116	test: 3.6504870	best: 3.6504870 (50)	total: 12.7s	remaining: 3m 56s
100:	learn: 2.3498792	test: 3.4297657	best: 3.4297657 (100)	total: 24.4s	remaining: 3m 37s
150:	learn: 1.9493453	test: 3.3283151	best: 3.3283151 (150)	total: 35.7s	remaining: 3m 20s
200:	learn: 1.6403552	test: 3.2653858	best: 3.2653858 (200)	total: 46.9s	remaining: 3m 6s
250:	learn: 1.3866464	test: 3.2258194	best: 3.2258194 (250)	total: 58.4s	remaining: 2m 54s
300:	learn: 1.1858036	test: 3.2012765	best: 3.2012765 (300)	total: 1m 9s	remaining: 2m 41s
350:	learn: 1.0072217	test: 3.1847127	best: 3.1847127 (350)	total: 1m 20s	remaining: 2m 29s
400:	learn: 0.8665772	test: 3.1777925	best: 3.1774433 (398)	total: 1m 32s	remaining: 2m 17s
450:	learn: 0.7536285	test: 3.1746805	best: 3.1746805 (450)	total: 1m 43s	remaining: 2m 5s
bestTest = 3.173861123
bestIteration = 469
Shrink model to first 470 iterations.

[Baseline] Ac

### Задание 4. Node2Vec с помощью библиотеки `node2vec`

Теперь построим представления узлов графа с помощью алгоритма Node2Vec.

Шаги:

1. Постройте граф `G` в формате `networkx.Graph` или `networkx.DiGraph`.
2. Используйте библиотеку `node2vec` (https://github.com/eliorc/node2vec) для генерации biased random walks и обучения эмбеддингов.
3. Получите матрицу эмбеддингов `Z` (num_nodes x embedding_dim).
4. Постройте модель классификации узлов по эмбеддингам (логрег / MLP / CatBoost).
5. Оцените качество на `test` и сохраните метрики.


In [18]:
G = nx.Graph()
G.add_nodes_from(range(num_papers))

edges = list(zip(edge_index_paper[0], edge_index_paper[1]))
G.add_edges_from(edges)
print(f"Граф создан. Узлов: {G.number_of_nodes()}, Рёбер: {G.number_of_edges()}")

n2v_obj = Node2Vec(
    G, 
    dimensions=64,
    walk_length=20,
    num_walks=30,
    workers=1,
    p=1.0, q=1.0,
    quiet=False
)

model_n2v = n2v_obj.fit(
    window=5, 
    min_count=1, 
    batch_words=4,
    workers=1
)


Z_node2vec = np.zeros((num_papers, 64))
for i in range(num_papers):
    Z_node2vec[i] = model_n2v.wv[str(i)]

del G
del n2v_obj
gc.collect()

clf_n2v = LogisticRegression(max_iter=150, solver='lbfgs', n_jobs=-1, verbose=1)

# Обучаем модель на вычисленных эмбеддингах
clf_n2v.fit(Z_node2vec[X_train_idx], y_train)

# Предсказание
y_pred_n2v = clf_n2v.predict(Z_node2vec[X_test_idx])

acc_n2v = accuracy_score(y_test, y_pred_n2v)
f1_macro_n2v = f1_score(y_test, y_pred_n2v, average='macro')

print(f"\n[Node2Vec] Accuracy на test: {acc_n2v:.4f}")
print(f"[Node2Vec] Macro-F1 на test: {f1_macro_n2v:.4f}")

Граф создан. Узлов: 30000, Рёбер: 9694


Computing transition probabilities:   0%|          | 0/30000 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 30/30 [00:14<00:00,  2.02it/s]



[Node2Vec] Accuracy на test: 0.0985
[Node2Vec] Macro-F1 на test: 0.0492


### Задание 5. Модификация обхода графа

В этом задании вам необходимо **модифицировать стратегию обхода графа** (random walk), чтобы улучшить качество генерируемых траекторий для обучения эмбеддингов.

Возможные направления модификаций:

* Использовать **biased random walks**, как в Node2Vec (параметры $p$ и $q$ управляют склонностью гулять локально или делать переходы "подальше").
* Делать переходы с вероятностями, зависящими от **степени** вершины или её **PageRank**.
* Вводить вероятности телепортации (random walk with restart).
* Использовать информацию о классах/кластеризации для более частого посещения похожих вершин.

Полезные статьи/источники для вдохновения:

* Node2Vec: Scalable Feature Learning for Networks — идея biased random walks: https://snap.stanford.edu/node2vec/
* DeepWalk: Online Learning of Social Representations — классические случайные обходы: https://arxiv.org/abs/1403.6652
* Role2Vec: Node embeddings для ролей (role-based random walks): https://arxiv.org/abs/1802.02896
* Обзор random walk методов и их связи с PageRank: https://link.springer.com/chapter/10.1007/978-3-319-17290-3_1

Вам нужно:

1. Реализовать свою функцию генерации траекторий `generate_custom_walks(G, ...)`, которая возвращает коллекцию последовательностей узлов.
2. Обосновать, почему выбранная стратегия может давать более полезные траектории, чем простой случайный обход.
3. Использовать эти траектории для обучения новой модели эмбеддингов.


**Модификация:** Degree-Biased Random Walk.
**Обоснование:** В сетях цитирования переходы к статьям-хабам (с высокой цитируемостью) дают больше информации о глобальной принадлежности статьи к классу (журналу), чем переходы к малоизвестным статьям. Вероятность перехода пропорциональна логарифму степени вершины.

Реализация выполнена через `scipy.sparse.csr_matrix` (сжатое хранение строкой) для достижения **максимальной скорости** генерации траекторий на CPU.

In [19]:
def generate_custom_walks(edge_index: np.ndarray, num_nodes: int, walk_length: int = 10, num_walks: int = 3) -> List[List[str]]:
    row = np.concatenate([edge_index[0], edge_index[1]])
    col = np.concatenate([edge_index[1], edge_index[0]])
    data = np.ones(len(row))
    
    adj = csr_matrix((data, (row, col)), shape=(num_nodes, num_nodes))
    
    degrees = np.array(adj.sum(axis=1)).flatten()
    
    node_weights = np.log(degrees + np.e)
    
    walks =[]

    for u in tqdm(range(num_nodes), desc="Walks generated"):
        for _ in range(num_walks):
            curr_node = u
            walk = [str(curr_node)]
            
            for _ in range(walk_length - 1):
                start_idx = adj.indptr[curr_node]
                end_idx = adj.indptr[curr_node + 1]
                neighbors = adj.indices[start_idx:end_idx]
                
                if len(neighbors) == 0:
                    break
                
                weights = node_weights[neighbors]
                probs = weights / weights.sum()
                
                next_node = np.random.choice(neighbors, p=probs)
                walk.append(str(next_node))
                curr_node = next_node
                
            walks.append(walk)
            
    return walks

custom_walks = generate_custom_walks(edge_index_paper, num_papers, walk_length=100, num_walks=30)

Walks generated: 100%|██████████| 30000/30000 [04:42<00:00, 106.36it/s]


### Задание 6. Выбор модели из NLP для обучения на траекториях

Используя траектории узлов, полученные в задании 5, выберите **модель из NLP**, чтобы обучить эмбеддинги узлов.

Возможные варианты моделей:

1. **Word2Vec / Skip-gram / CBOW** (Gensim):
   * https://radimrehurek.com/gensim/models/word2vec.html
   * Варианты: изменить размерность, окно, negative sampling, sg=0/1.

2. **FastText** (Gensim) — учитывает подслова, можно попробовать, если вы хотите более "гладкие" эмбеддинги:
   * https://radimrehurek.com/gensim/models/fasttext.html

3. **Простая sequence-модель на PyTorch** (RNN / GRU / LSTM / Transformer):
   * идея: обучить модель предсказывать следующую вершину в траектории,
     а внутренние представления скрытого слоя/энкодера использовать как эмбеддинги узлов.
   * полезные ссылки:
     * Туториал по языковому моделированию в PyTorch: https://pytorch.org/tutorials/beginner/transformer_tutorial.html
     * Примеры seq2seq моделей: https://pytorch.org/tutorials/

Требования к заданию:

* Чётко опишите, **какую модель** вы выбрали и **почему**.
* Обучите модель на траекториях из задания 5.
* Получите матрицу эмбеддингов узлов (например, усреднив по позициям, где узел встречается, или взяв эмбеддинг слоя embedding / encoder).
* Используйте эти эмбеддинги для решения задачи классификации узлов (аналогично шагу с Node2Vec).


In [20]:
from gensim.models import FastText

ft_model = FastText(
    sentences=custom_walks, 
    vector_size=64, 
    window=5, 
    min_count=1, 
    sg=1, 
    workers=15,
    epochs=5 
)


Z_custom = np.zeros((num_papers, 64))
for i in range(num_papers):
    Z_custom[i] = ft_model.wv[str(i)]
    

gc.collect()

clf_custom = LogisticRegression(max_iter=150,  solver='lbfgs', n_jobs=-1)
clf_custom.fit(Z_custom[X_train_idx], y_train)


y_pred_custom = clf_custom.predict(Z_custom[X_test_idx])

acc_custom = accuracy_score(y_test, y_pred_custom)
f1_macro_custom = f1_score(y_test, y_pred_custom, average='macro')

print(f"\n[Custom + FastText] Accuracy на test: {acc_custom:.4f}")
print(f"[Custom + FastText] Macro-F1 на test: {f1_macro_custom:.4f}")


[Custom + FastText] Accuracy на test: 0.0868
[Custom + FastText] Macro-F1 на test: 0.0155


### Задание 7. Сравнение метрик baseline, Node2Vec и кастомного метода

Сравните качество трёх подходов на тестовом множестве:

1. Бейзлайн без графа (CatBoost по исходным признакам).
2. Node2Vec + классификатор по эмбеддингам.
3. Кастомный обход + выбранная NLP-модель + классификатор по эмбеддингам.

Рекомендуется оформить результаты в виде таблицы и кратко их прокомментировать (1–2 абзаца текста).


In [21]:
results = {
    "Метод":[
        "1. Бейзлайн (CatBoost по исходным текстам)", 
        "2. Node2Vec (Случайный обход + W2V)", 
        "3. Кастомный (Degree-biased обход + FastText)"
    ],
    "Accuracy":[acc_baseline, acc_n2v, acc_custom],
    "Macro-F1":[f1_macro_baseline, f1_macro_n2v, f1_macro_custom]
}

df_results = pd.DataFrame(results)

styled_df = df_results.style.format({
    'Accuracy': '{:.4f}',
    'Macro-F1': '{:.4f}'
}).set_properties(**{'text-align': 'center'})

display(styled_df)

,Метод,Accuracy,Macro-F1
0,1. Бейзлайн (CatBoost по исходным текстам),0.2370,0.0882
1,2. Node2Vec (Случайный обход + W2V),0.0985,0.0492
2,3. Кастомный (Degree-biased обход + FastText),0.0868,0.0155
